In [66]:
# -*- coding: utf-8 -*-
# LangGraph 4-Tool Agent — Colab Ready
#
# Four capabilities:
# 1. DIRECT_TOOL
# 2. CALCULATOR_TOOL
# 3. RAG_TOOL — PDF only, with OCR fallback
# 4. TEXT_TO_IMAGE_TOOL — Qwen2.5-VL prompt enhancement + Stable Diffusion
#
# Designed for a Google Colab GPU runtime.


In [67]:
# CELL 1 — INSTALL REQUIRED LIBRARIES

!pip install -q -U transformers accelerate sentence-transformers \
    langchain langchain-community langchain-huggingface \
    langchain-text-splitters langgraph faiss-cpu pypdf pymupdf \
    pytesseract huggingface_hub gradio diffusers safetensors ftfy \
    qwen-vl-utils

!apt-get -qq update
!apt-get -qq install -y tesseract-ocr

print("Dependencies installed.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 64.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.8/739.8 kB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.5/161.5 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 61.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 49.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.8/393.8 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 798.3/798.3 kB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.

In [68]:
# CELL 3: IMPORT LIBRARIES
# ============================================================

# Import PyTorch for running the language model.
import torch

# Import Transformers classes for loading Gemma.
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    pipeline,
)

# Import Hugging Face embeddings.
from langchain_huggingface import HuggingFaceEmbeddings

# Import FAISS vector database.
from langchain_community.vectorstores import FAISS

# Import text splitter.
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Import LangChain Document object.
from langchain_core.documents import Document

# Import PyMuPDF for reading/rendering PDFs.
import fitz

# Import PIL for handling page images.
from PIL import Image

# Import Tesseract OCR.
import pytesseract

# Import Google Colab file uploader.

# Import Python's mathematical tools.
import math

# Import regular expressions.
import re

print("Libraries imported successfully.")


/tmp/ipykernel_1492/1026747431.py:18: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Libraries imported successfully.


In [ ]:
# CELL 9: LOAD QWEN 2.5 1.5B INSTRUCT
# ============================================================
# Qwen2.5-1.5B-Instruct is used as our LLM.
#
# We use an instruction-tuned model because it is better suited
# to following RAG prompts and later making agent decisions.
#
# The Hugging Face model card recommends using recent
# Transformers versions and supports direct loading with
# AutoTokenizer and AutoModelForCausalLM.
# ============================================================

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    pipeline,
)

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"


# ------------------------------------------------------------
# Load tokenizer
# ------------------------------------------------------------

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)


# ------------------------------------------------------------
# Load model
# ------------------------------------------------------------

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto"
)


# ------------------------------------------------------------
# Check device
# ------------------------------------------------------------

print("Model loaded successfully.")

print(
    "Model device:",
    model.device
)

# ============================================================
# CELL 10: CREATE QWEN GENERATION FUNCTION
# ============================================================
# This function sends a system instruction and user prompt
# to Qwen and returns only the newly generated answer.
# ============================================================

def generate_response(
    user_prompt,
    system_prompt="You are a helpful AI assistant."
):

    # --------------------------------------------------------
    # Create chat messages.
    # --------------------------------------------------------

    messages = [
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": user_prompt
        }
    ]


    # --------------------------------------------------------
    # Convert messages into Qwen's chat format.
    # --------------------------------------------------------

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )


    # --------------------------------------------------------
    # Tokenize the prompt.
    # --------------------------------------------------------

    inputs = tokenizer(
        [text],
        return_tensors="pt"
    ).to(model.device)


    # --------------------------------------------------------
    # Generate response.
    # --------------------------------------------------------

    generated_ids = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.2,
        do_sample=False
    )


    # --------------------------------------------------------
    # Remove input tokens from generated output.
    # --------------------------------------------------------

    generated_ids = [
        output_ids[len(input_ids):]
        for input_ids, output_ids
        in zip(
            inputs.input_ids,
            generated_ids
        )
    ]


    # --------------------------------------------------------
    # Convert tokens back into text.
    # --------------------------------------------------------

    response = tokenizer.batch_decode(
        generated_ids,
        skip_special_tokens=True
    )[0]


    return response.strip()

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

In [ ]:
!pip uninstall -y clip
!pip install git+https://github.com/openai/CLIP.git
import clip
# CELL 3.2: IMPORT STABLE DIFFUSION LIBRARIES
# ============================================================
import torch
from torch import autocast
from diffusers import StableDiffusionPipeline
from PIL import Image
import os # For saving images

# ============================================================
# CELL 3.3: LOAD STABLE DIFFUSION AND CLIP MODELS
# ============================================================

# Load Stable Diffusion pipeline
print("Loading Stable Diffusion pipeline...")
model_id = "runwayml/stable-diffusion-v1-5"
pipe = StableDiffusionPipeline.from_pretrained(model_id, torch_dtype=torch.float16)
pipe = pipe.to("cuda")
print("Stable Diffusion pipeline loaded.")

# Load CLIP model
print("Loading CLIP model...")
device = "cuda" if torch.cuda.is_available() else "cpu"
clip_model, preprocess = clip.load("ViT-B/32", device=device)
print("CLIP model loaded.")

# ============================================================
# VISION-LANGUAGE MODEL + TEXT-TO-IMAGE
# ============================================================
# IMPORTANT:
# A Vision-Language Model (VLM) understands/rewrites multimodal
# prompts; the diffusion model is the component that actually
# generates the image.
#
# Pipeline:
# User text prompt
#       ↓
# Qwen2.5-VL-3B-Instruct (prompt understanding/enhancement)
#       ↓
# Stable Diffusion
#       ↓
# PNG file
#       ↓
# Gradio Image component
# ============================================================

import os
import uuid
import torch
from PIL import Image
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

VLM_NAME = "Qwen/Qwen2.5-VL-3B-Instruct"

print("Loading Vision-Language Model...")
vlm_processor = AutoProcessor.from_pretrained(VLM_NAME)

vlm_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    VLM_NAME,
    torch_dtype="auto",
    device_map="auto"
)

print("VLM loaded:", VLM_NAME)


def enhance_image_prompt(user_prompt: str) -> str:
    """
    Use the VLM to convert a short user request into a detailed,
    image-generation-friendly prompt. """
    messages = [
        {
            "role": "system",
            "content": (
                "You are an image prompt enhancement assistant. "
                "Convert the user's request into one detailed prompt "
                "for a Stable Diffusion image generator. "
                "Include subject, environment, composition, lighting, "
                "camera/style details when useful. "
                "Return ONLY the final image prompt."
            )
        },
        {
            "role": "user",
            "content": user_prompt
        }
    ]

    try:
        text = vlm_processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = vlm_processor(
            text=[text],
            return_tensors="pt"
        )

        # Move tensors to the same device as the VLM.
        inputs = {
            k: v.to(vlm_model.device)
            if hasattr(v, "to") else v
            for k, v in inputs.items()
        }

        with torch.inference_mode():
            output_ids = vlm_model.generate(
                **inputs,
                max_new_tokens=180,
                do_sample=False
            )

        # Remove the prompt tokens.
        generated = output_ids[0][inputs["input_ids"].shape[1]:]

        enhanced = vlm_processor.tokenizer.decode(
            generated,
            skip_special_tokens=True
        ).strip()

        return enhanced if enhanced else user_prompt

    except Exception as e:
        print("VLM prompt enhancement failed:", e)
        print("Using original prompt instead.")
        return user_prompt


IMAGE_DIR = "generated_images"
os.makedirs(IMAGE_DIR, exist_ok=True)


def text_to_image_tool(prompt):
    """
    VLM-enhanced text-to-image generation.
    Returns a dictionary so the LangGraph state can expose the
    generated image path to Gradio.
    """
    try:
        enhanced_prompt = enhance_image_prompt(str(prompt))

        print("\nOriginal prompt:")
        print(prompt)

        print("\nVLM enhanced prompt:")
        print(enhanced_prompt)

        # Stable Diffusion is the actual image generator.
        result = pipe(
            enhanced_prompt,
            guidance_scale=8.5,
            num_inference_steps=30
        )

        image = result.images[0]

        path = os.path.abspath(
            os.path.join(
                IMAGE_DIR,
                f"{uuid.uuid4().hex}.png"
            )
        )

        image.save(path, format="PNG")

        return {
            "tool": "TEXT_TO_IMAGE_TOOL",
            "answer": "Image generated successfully.",
            "image_path": path,
            "prompt": enhanced_prompt
        }

    except Exception as e:
        return {
            "tool": "TEXT_TO_IMAGE_TOOL",
            "answer": f"Text-to-image generation error: {e}",
            "image_path": None,
            "prompt": str(prompt)
        }

print("VLM + Stable Diffusion text-to-image pipeline ready.")

In [ ]:
# CELL 4 — EMBEDDINGS

from langchain_community.document_loaders import PyPDFLoader
from langchain_huggingface import HuggingFaceEmbeddings

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL
)

print("Embedding model loaded:", EMBEDDING_MODEL)

### ⚠️ **Important: Resolve NameError by running cells in order**

To ensure the agent functions correctly and avoid `NameError` issues, please execute the following cells in the specified order:

1.  **Cell 625a2111**: Defines the Stable Diffusion and CLIP models, including the `text_to_image_tool` function.
2.  **Cell 4688d9df**: Loads the Qwen LLM and defines the `generate_response` function.
3.  **Cell 75b59840**: Defines the Multimodal RAG functions, which depend on `generate_response`.
4.  **Cell cfce6fc0** (or **Cell 48282d45** if you are using that duplicate for tools): Defines the agent's tools, which rely on `generate_response`, `text_to_image_tool`, and RAG functions.
5.  **Cell cd04bb04**: Sets up the LangGraph workflow.
6.  **Cell 19cdf535**: Defines the `run_langgraph` function.
7.  **Cell 8c1e59c2**: Launches the Gradio user interface.

In [ ]:
# CELL 9: LOAD QWEN 2.5 1.5B INSTRUCT
# ============================================================
# Qwen2.5-1.5B-Instruct is used as our LLM.
#
# We use an instruction-tuned model because it is better suited
# to following RAG prompts and later making agent decisions.
#
# The Hugging Face model card recommends using recent
# Transformers versions and supports direct loading with
# AutoTokenizer and AutoModelForCausalLM.
# ============================================================

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    pipeline,
)

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"


# ------------------------------------------------------------
# Load tokenizer
# ------------------------------------------------------------

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)


# ------------------------------------------------------------
# Load model
# ------------------------------------------------------------

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto"
)


# ------------------------------------------------------------
# Check device
# ------------------------------------------------------------

print("Model loaded successfully.")

print(
    "Model device:",
    model.device
)

# ============================================================
# CELL 10: CREATE QWEN GENERATION FUNCTION
# ============================================================
# This function sends a system instruction and user prompt
# to Qwen and returns only the newly generated answer.
# ============================================================

def generate_response(
    user_prompt,
    system_prompt="You are a helpful AI assistant."
):

    # --------------------------------------------------------
    # Create chat messages.
    # --------------------------------------------------------

    messages = [
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": user_prompt
        }
    ]


    # --------------------------------------------------------
    # Convert messages into Qwen's chat format.
    # --------------------------------------------------------

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )


    # --------------------------------------------------------
    # Tokenize the prompt.
    # --------------------------------------------------------

    inputs = tokenizer(
        [text],
        return_tensors="pt"
    ).to(model.device)


    # --------------------------------------------------------
    # Generate response.
    # --------------------------------------------------------

    generated_ids = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.2,
        do_sample=False
    )


    # --------------------------------------------------------
    # Remove input tokens from generated output.
    # --------------------------------------------------------

    generated_ids = [
        output_ids[len(input_ids):]
        for input_ids, output_ids
        in zip(
            inputs.input_ids,
            generated_ids
        )
    ]


    # --------------------------------------------------------
    # Convert tokens back into text.
    # --------------------------------------------------------

    response = tokenizer.batch_decode(
        generated_ids,
        skip_special_tokens=True
    )[0]


    return response.strip()

In [ ]:
# ============================================================
# MULTIMODAL RAG — PDF + IMAGE
# ============================================================
# Any uploaded PDF or image is automatically converted to text
# and indexed in the same FAISS RAG database.
#
# PDF:
#   1. Extract native PDF text
#   2. OCR scanned/image-only pages when necessary
#
# IMAGE:
#   1. OCR the image
#   2. Store the extracted text in RAG
#
# The same uploaded file can therefore be:
#   upload -> extract text -> chunk -> embed -> FAISS -> RAG
# ============================================================

CURRENT_VECTOR_DB = None
CURRENT_SOURCE_NAME = None
CURRENT_SOURCE_TYPE = None
CURRENT_EXTRACTED_TEXT = ""


def ocr_image_file(file_path):
    """Extract text from a standalone image using Tesseract OCR."""
    image = Image.open(file_path).convert("RGB")
    return pytesseract.image_to_string(image, lang="eng").strip()


def extract_pdf_documents(file_path):
    """Extract PDF text with OCR fallback for scanned pages."""
    loader = PyPDFLoader(file_path)
    docs = loader.load()

    extracted_chars = sum(
        len(d.page_content.strip()) for d in docs
    )

    # If the PDF has little/no native text, OCR each page.
    if extracted_chars < 500:
        docs = []
        pdf = fitz.open(file_path)

        for page_number, page in enumerate(pdf):
            pix = page.get_pixmap(
                matrix=fitz.Matrix(2, 2),
                alpha=False
            )

            image = Image.frombytes(
                "RGB",
                [pix.width, pix.height],
                pix.samples
            )

            text = pytesseract.image_to_string(
                image,
                lang="eng"
            ).strip()

            docs.append(
                Document(
                    page_content=text,
                    metadata={
                        "source": file_path,
                        "page": page_number + 1,
                        "source_type": "pdf_ocr"
                    }
                )
            )

        pdf.close()

    return docs


def load_uploaded_file(file_path):
    """Index either a PDF or an image into the current RAG database."""
    global CURRENT_VECTOR_DB
    global CURRENT_SOURCE_NAME
    global CURRENT_SOURCE_TYPE
    global CURRENT_EXTRACTED_TEXT

    if not file_path:
        return "No file selected."

    ext = os.path.splitext(file_path)[1].lower()

    supported_images = {
        ".png", ".jpg", ".jpeg", ".webp", ".bmp", ".tiff", ".tif"
    }

    docs = []

    try:
        # ----------------------------------------------------
        # PDF
        # ----------------------------------------------------
        if ext == ".pdf":
            docs = extract_pdf_documents(file_path)
            source_type = "PDF"

        # ----------------------------------------------------
        # IMAGE
        # ----------------------------------------------------
        elif ext in supported_images:
            text = ocr_image_file(file_path)

            docs = [
                Document(
                    page_content=text,
                    metadata={
                        "source": file_path,
                        "page": 1,
                        "source_type": "image_ocr"
                    }
                )
            ]
            source_type = "IMAGE"

        else:
            return (
                "Unsupported file type. "
                "Upload a PDF or image (PNG/JPG/JPEG/WEBP/BMP/TIFF)."
            )

        if not docs:
            return "No readable content was found."

        readable_chars = sum(
            len(d.page_content.strip()) for d in docs
        )

        if readable_chars == 0:
            return (
                f"{source_type} uploaded, but no readable text "
                "was found by text extraction/OCR."
            )

        # ----------------------------------------------------
        # Split extracted text
        # ----------------------------------------------------
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=700,
            chunk_overlap=100
        )

        upload_chunks = splitter.split_documents(docs)

        if not upload_chunks:
            return "Could not create searchable chunks."

        # ----------------------------------------------------
        # Build FAISS
        # ----------------------------------------------------
        CURRENT_VECTOR_DB = FAISS.from_documents(
            documents=upload_chunks,
            embedding=embeddings
        )

        CURRENT_SOURCE_NAME = os.path.basename(file_path)
        CURRENT_SOURCE_TYPE = source_type
        CURRENT_EXTRACTED_TEXT = "\n\n".join(
            d.page_content for d in docs
        )

        pages = len(docs)

        return (
            f"{source_type} indexed successfully: "
            f"{CURRENT_SOURCE_NAME}\n"
            f"Pages/sections: {pages}\n"
            f"Characters extracted: {readable_chars}\n"
            f"RAG chunks: {len(upload_chunks)}"
        )

    except Exception as e:
        CURRENT_VECTOR_DB = None
        CURRENT_SOURCE_NAME = None
        CURRENT_SOURCE_TYPE = None
        CURRENT_EXTRACTED_TEXT = ""

        return f"File indexing error: {type(e).__name__}: {e}"


def rag_for_current_upload(question):
    """Answer using only the currently uploaded PDF/image RAG index."""
    global CURRENT_VECTOR_DB

    if CURRENT_VECTOR_DB is None:
        return {
            "answer": (
                "No PDF or image is indexed yet. "
                "Upload a file first."
            ),
            "documents": []
        }

    retrieved_docs = CURRENT_VECTOR_DB.similarity_search(
        question,
        k=4
    )

    if not retrieved_docs:
        return {
            "answer": (
                "I could not find relevant information "
                "in the uploaded file."
            ),
            "documents": []
        }

    context_parts = []

    for doc in retrieved_docs:
        page = doc.metadata.get("page", "unknown")
        context_parts.append(
            f"[Page/Section {page}]\n{doc.page_content}"
        )

    context = "\n\n".join(context_parts)

    prompt = f"""
Answer the user's question using ONLY the extracted content
from the uploaded file below.

Uploaded file: {CURRENT_SOURCE_NAME}
File type: {CURRENT_SOURCE_TYPE}

DOCUMENT CONTENT:
{context}

USER QUESTION:
{question}

Rules:
- Do not invent facts that are not supported by the retrieved content.
- If the answer is not present, say that it was not found in the uploaded file.
- Give a clear, useful text answer.
"""

    answer = generate_response(
        prompt,
        system_prompt=(
            "You are a document RAG assistant. "
            "Ground every answer in the supplied document context."
        )
    )

    return {
        "answer": answer,
        "documents": retrieved_docs
    }


print("Multimodal PDF + Image RAG ready.")


In [ ]:
from langchain_core.tools import tool

# ============================================================
# FIVE AGENT TOOLS
# ============================================================
import re

@tool
def direct_tool(question: str) -> str:
    """Answer a normal general-purpose question using the LLM."""
    return generate_response(
        question,
        system_prompt=(
            "You are a helpful general-purpose AI assistant. "
            "Answer clearly and concisely."
        )
    )


@tool
def calculator_tool(expression: str) -> str:
    """Calculate a basic arithmetic expression."""
    try:
        expression = str(expression).strip()

        if not re.fullmatch(
            r"[0-9+\-*/().%\s]+",
            expression
        ):
            return "Invalid mathematical expression."

        result = eval(
            expression,
            {"__builtins__": {}},
            {}
        )

        return str(result)

    except Exception as e:
        return f"Calculation error: {e}"


@tool
def rag_tool(question: str) -> str:
    """Answer using the currently indexed PDF or image."""
    result = rag_for_current_upload(question)

    pages = list(dict.fromkeys(
        str(d.metadata.get("page", "unknown"))
        for d in result.get("documents", [])
    ))

    answer = result.get("answer", "")

    if pages:
        answer += (
            "\n\nSource Pages/Sections: "
            + ", ".join(pages)
        )

    return answer


@tool
def image_to_text_tool(image_path: str) -> str:
    """
    Extract text from an uploaded image using OCR.
    The same image is also indexed in RAG when uploaded.
    """
    try:
        text = ocr_image_file(image_path)

        if not text.strip():
            return (
                "No readable text was detected in the image."
            )

        return (
            "Extracted text from image:\n\n"
            + text
        )

    except Exception as e:
        return f"Image-to-text error: {type(e).__name__}: {e}"


@tool
def text_to_image_tool_bound(prompt: str) -> str:
    """Generate an image from a text prompt."""
    result = text_to_image_tool(prompt)

    if result["image_path"]:
        return (
            "Image generated successfully.\n"
            f"IMAGE_PATH::{result['image_path']}"
        )

    return result["answer"]


TOOLS = [
    direct_tool,
    calculator_tool,
    rag_tool,
    image_to_text_tool,
    text_to_image_tool_bound
]

TOOL_NAMES = [t.name for t in TOOLS]

print("Five tools:", TOOL_NAMES)


In [ ]:
# ============================================================
# MULTIMODAL TOOL ROUTER
# ============================================================
import re

def is_image_generation_request(question: str) -> bool:
    q = re.sub(r"\s+", " ", question.lower().strip())

    patterns = [
        r"\bgenerate\b.*\bimage\b",
        r"\bcreate\b.*\bimage\b",
        r"\bmake\b.*\bimage\b",
        r"\bdraw\b.*\bimage\b",
        r"\bgenerate\b.*\bpicture\b",
        r"\bcreate\b.*\bpicture\b",
        r"\bmake\b.*\bpicture\b",
        r"\bdraw\b.*\bpicture\b",
        r"\btext[- ]to[- ]image\b",
        r"\bimage of\b",
        r"\bpicture of\b",
        r"\bvisualize\b",
    ]

    return any(re.search(p, q) for p in patterns)


def is_image_text_request(question: str) -> bool:
    q = question.lower()

    patterns = [
        "extract text from image",
        "read the text in image",
        "ocr this image",
        "convert image to text",
        "image to text",
        "read this image",
        "what text is in the image",
        "transcribe the image",
    ]

    return any(x in q for x in patterns)


def choose_tool_fallback(question: str, has_uploaded_file=False,
                         uploaded_file_type=None) -> str:

    # Image generation must be checked first.
    if is_image_generation_request(question):
        return "text_to_image_tool_bound"

    # Explicit image-to-text request.
    if (
        uploaded_file_type == "IMAGE"
        and is_image_text_request(question)
    ):
        return "image_to_text_tool"

    # Pure arithmetic.
    if re.fullmatch(
        r"[0-9+\-*/().%\s]+",
        question.strip()
    ):
        return "calculator_tool"

    # If a file is uploaded, document questions go to RAG.
    # This makes an uploaded image available to RAG as well.
    if has_uploaded_file:
        return "rag_tool"

    # Explicit RAG/document language.
    rag_words = [
        "according to the document",
        "according to the uploaded",
        "uploaded document",
        "uploaded file",
        "document",
        "pdf",
        "image",
        "syllabus",
        "curriculum",
        "regulation",
        "rules in the document",
        "what does the document say",
        "what does the file say",
    ]

    if any(x in question.lower() for x in rag_words):
        return "rag_tool"

    return "direct_tool"


print("Multimodal tool router ready.")


In [ ]:
from langchain_core.tools import tool

@tool
def direct_tool(question: str) -> str:
    """Answer a normal general-purpose question using the LLM."""
    return generate_response(
        question,
        system_prompt=(
            "You are a helpful general-purpose AI assistant. "
            "Answer clearly and concisely."
        )
    )


@tool
def calculator_tool(expression: str) -> str:
    """Calculate a basic arithmetic expression."""
    try:
        expression = str(expression).strip()

        if not re.fullmatch(
            r"[0-9+\-*/().%\s]+",
            expression
        ):
            return "Invalid mathematical expression."

        result = eval(
            expression,
            {"__builtins__": {}},
            {}
        )

        return str(result)

    except Exception as e:
        return f"Calculation error: {e}"


@tool
def rag_tool(question: str) -> str:
    """Answer using the currently indexed PDF or image."""
    result = rag_for_current_upload(question)

    pages = list(dict.fromkeys(
        str(d.metadata.get("page", "unknown"))
        for d in result.get("documents", [])
    ))

    answer = result.get("answer", "")

    if pages:
        answer += (
            "\n\nSource Pages/Sections: "
            + ", ".join(pages)
        )

    return answer


@tool
def image_to_text_tool(image_path: str) -> str:
    """
    Extract text from an uploaded image using OCR.
    The same image is also indexed in RAG when uploaded.
    """
    try:
        text = ocr_image_file(image_path)

        if not text.strip():
            return (
                "No readable text was detected in the image."
            )

        return (
            "Extracted text from image:\n\n"
            + text
        )

    except Exception as e:
        return f"Image-to-text error: {type(e).__name__}: {e}"


@tool
def text_to_image_tool_bound(prompt: str) -> str:
    """Generate an image from a text prompt."""
    result = text_to_image_tool(prompt)

    if result["image_path"]:
        return (
            "Image generated successfully.\n"
            f"IMAGE_PATH::{result['image_path']}"
        )

    return result["answer"]


TOOLS = [
    direct_tool,
    calculator_tool,
    rag_tool,
    image_to_text_tool,
    text_to_image_tool_bound
]

TOOL_NAMES = [t.name for t in TOOLS]

print("Five tools (re-defined):", TOOL_NAMES)

In [ ]:
from typing import TypedDict, List, Optional, Dict, Any
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import HumanMessage


class AgentState(TypedDict):
    question: str
    selected_tool: str
    tool_result: str
    source_pages: List[str]
    image_path: Optional[str]
    final_answer: str
    history: List[Dict[str, str]]
    messages: List[Any]
    uploaded_file_path: Optional[str]
    uploaded_file_type: Optional[str]


def agent_node(state: AgentState):
    question = state["question"]

    selected_tool = choose_tool_fallback(
        question,
        has_uploaded_file=bool(state.get("uploaded_file_path")),
        uploaded_file_type=state.get("uploaded_file_type")
    )

    print("[Agent] Selected:", selected_tool)

    return {
        "selected_tool": selected_tool,
        "messages": [HumanMessage(content=question)]
    }


def direct_node(state: AgentState):
    result = direct_tool.invoke(state["question"])

    return {
        "tool_result": str(result),
        "source_pages": [],
        "image_path": None
    }


def calculator_node(state: AgentState):
    result = calculator_tool.invoke(state["question"])

    return {
        "tool_result": str(result),
        "source_pages": [],
        "image_path": None
    }


def rag_node(state: AgentState):
    result = rag_for_current_upload(state["question"])

    pages = list(dict.fromkeys(
        str(doc.metadata.get("page", "unknown"))
        for doc in result.get("documents", [])
    ))

    return {
        "tool_result": result.get("answer", ""),
        "source_pages": pages,
        "image_path": None
    }


def image_to_text_node(state: AgentState):
    image_path = state.get("uploaded_file_path")

    if not image_path:
        return {
            "tool_result": "Please upload an image first.",
            "source_pages": [],
            "image_path": None
        }

    result = image_to_text_tool.invoke(image_path)

    return {
        "tool_result": str(result),
        "source_pages": [],
        "image_path": None
    }


def text_to_image_node(state: AgentState):
    result = text_to_image_tool(state["question"])

    return {
        "tool_result": result.get(
            "answer",
            "Image generation failed."
        ),
        "source_pages": [],
        "image_path": result.get("image_path")
    }


def route_tool(state: AgentState):
    routes = {
        "direct_tool": "direct",
        "calculator_tool": "calculator",
        "rag_tool": "rag",
        "image_to_text_tool": "image_to_text",
        "text_to_image_tool_bound": "text_to_image",
    }

    return routes.get(
        state.get("selected_tool"),
        "direct"
    )


def final_answer_node(state: AgentState):
    answer = state.get("tool_result", "")

    if state.get("selected_tool") == "calculator_tool":
        answer = f"**Result:** {answer}"

    history = list(state.get("history", []))

    history.extend([
        {
            "role": "user",
            "content": state["question"]
        },
        {
            "role": "assistant",
            "content": answer
        }
    ])

    return {
        "final_answer": answer,
        "history": history
    }


# ============================================================
# BUILD LANGGRAPH
# ============================================================

workflow = StateGraph(AgentState)

workflow.add_node("agent", agent_node)
workflow.add_node("direct", direct_node)
workflow.add_node("calculator", calculator_node)
workflow.add_node("rag", rag_node)
workflow.add_node("image_to_text", image_to_text_node)
workflow.add_node("text_to_image", text_to_image_node)
workflow.add_node("final", final_answer_node)

workflow.add_edge(START, "agent")

workflow.add_conditional_edges(
    "agent",
    route_tool,
    {
        "direct": "direct",
        "calculator": "calculator",
        "rag": "rag",
        "image_to_text": "image_to_text",
        "text_to_image": "text_to_image",
    }
)

for node in [
    "direct",
    "calculator",
    "rag",
    "image_to_text",
    "text_to_image"
]:
    workflow.add_edge(node, "final")

workflow.add_edge("final", END)

memory = MemorySaver()
app = workflow.compile(checkpointer=memory)

print("LangGraph 5-tool multimodal agent compiled successfully.")


In [ ]:
# CELL 11 — GRAPH VERIFICATION

print("Nodes:", list(workflow.nodes))
print("Tools:", TOOL_NAMES)
print("Multimodal routing: PDF + Image + Text + Arithmetic + Text-to-Image")


In [ ]:
# ============================================================
# GRAPH RUNNER
# ============================================================

def sanitize_history(history):
    safe = []

    for item in history or []:
        if isinstance(item, dict):
            role = item.get("role")
            content = item.get("content")

            if isinstance(role, str) and isinstance(content, str):
                safe.append({
                    "role": role,
                    "content": content
                })

        elif isinstance(item, (list, tuple)) and len(item) >= 2:
            if isinstance(item[0], str):
                safe.append({
                    "role": "user",
                    "content": item[0]
                })

            if isinstance(item[1], str):
                safe.append({
                    "role": "assistant",
                    "content": item[1]
                })

    return safe


def run_langgraph(
    question,
    thread_id="default",
    history=None,
    uploaded_file_path=None,
    uploaded_file_type=None
):
    state = {
        "question": str(question),
        "selected_tool": "",
        "tool_result": "",
        "source_pages": [],
        "image_path": None,
        "final_answer": "",
        "history": sanitize_history(history),
        "messages": [],
        "uploaded_file_path": uploaded_file_path,
        "uploaded_file_type": uploaded_file_type,
    }

    return app.invoke(
        state,
        config={
            "configurable": {
                "thread_id": thread_id
            }
        }
    )


print("Multimodal graph runner ready.")


In [ ]:
# ============================================================
# ORIGINAL GRADIO UI
# ============================================================
import gradio as gr
with gr.Blocks(
    title="Multimodal LangGraph Agent",
    fill_height=True,
) as demo:

    gr.Markdown(
        "# 🤖 Multimodal LangGraph Agent"
    )

    gr.Markdown(
        "Upload a **PDF/image** or ask a **text/arithmetic** question. "
        "The agent selects the appropriate tool automatically."
    )

    with gr.Row(equal_height=False):

        # ========================================================
        # LEFT SIDEBAR
        # ========================================================

        with gr.Column(
            scale=1,
            min_width=280
        ):

            gr.Markdown("### 📎 Files")

            rag_file = gr.File(
                label="Upload PDF or Image",
                file_types=SUPPORTED_FILE_TYPES,
                type="filepath",
            )

            index_status = gr.Textbox(
                label="RAG Status",
                placeholder=(
                    "Upload a PDF or image. "
                    "It will be automatically indexed."
                ),
                interactive=False,
                lines=7,
            )

            rag_file.change(
                fn=index_file_callback,
                inputs=rag_file,
                outputs=index_status,
            )

            gr.Markdown(
                """
**Supported**

- 📄 PDF
- 🏞️ PNG / JPG / JPEG / WEBP
- 🔢 Arithmetic
- 🗣 Normal text
- 🎨 Text-to-image

**Uploaded PDF/image flow**

`File → Text/OCR → Chunks → Embeddings → FAISS → RAG`
"""
            )

        # ========================================================
        # MAIN CHAT
        # ========================================================

        with gr.Column(scale=4):

            chatbot = gr.Chatbot(
                label="Agent",
                height=620,
            )

            generated_image = gr.Image(
                label="Generated Image",
                type="filepath",
                height=420,
            )

            # ====================================================
            # MESSAGE BOX
            # ====================================================

            with gr.Row():

                message = gr.Textbox(
                    placeholder=(
                        "Ask anything...  "
                        "e.g. 'summarize the uploaded file', "
                        "'extract text from this image', "
                        "'25 * 8', or "
                        "'generate an image of a dog'"
                    ),
                    lines=2,
                    scale=8,
                    show_label=False,
                )

                send_button = gr.Button(
                    "Send",
                    variant="primary",
                    scale=1,
                )

            clear_button = gr.Button(
                "Clear Chat",
                size="sm",
            )

    # ============================================================
    # EVENTS
    # ============================================================

    send_button.click(
        fn=gradio_chat,
        inputs=[
            message,
            chatbot,
            rag_file,
        ],
        outputs=[
            message,
            chatbot,
            generated_image,
        ],
    )

    message.submit(
        fn=gradio_chat,
        inputs=[
            message,
            chatbot,
            rag_file,
        ],
        outputs=[
            message,
            chatbot,
            generated_image,
        ],
    )

    clear_button.click(
        fn=lambda: ([], None),
        inputs=None,
        outputs=[
            chatbot,
            generated_image,
        ],
    )


# ============================================================
# LAUNCH
# ============================================================

print("Starting Multimodal LangGraph Agent...")


In [ ]:

demo.launch(
    share=True,
    debug=True,
)